# 08 · Documents: Send a PDF, or Build a Library 📄📚

Mei Ling and Ahmad are flat hunting. Their PDFs are in the **`docs`** folder — open a few in the file list:

| File | What's inside |
|---|---|
| `my_flat_notes.pdf` | what they want: budget, towns, size, lease |
| `listing_tampines.pdf`, `listing_bedok.pdf`, `listing_pasir_ris.pdf` | three flats from an agent |
| `viewing_notes.pdf` | what they thought at Saturday's viewings |
| `renovation_quotes.pdf` | two quotes from contractors |

Two ways to give PDFs to the AI:
- **Send the PDF** with your message — simple, good for **one** file.
- **A vector store** — a searchable library of **many** files. The AI looks up only the parts it needs.

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from openai import OpenAI

if not os.path.exists("main.py"):   # make sure we're in the project folder
    os.chdir("../..")

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

client = OpenAI()
MODEL = "gpt-6-luna"
print("Ready!")

In [ ]:
bot = "You are SG Data Bot, a helpful assistant for life in Singapore. Keep replies short."

## 1. What's in the folder?

In [ ]:
for name in os.listdir("docs"):
    print(name)

## 2. Way #1: send one PDF 📄
Upload it, then send it as an `input_file` together with your question.

In [ ]:
notes_file = client.files.create(file=open("docs/my_flat_notes.pdf", "rb"), purpose="user_data")

response = client.responses.create(
    model=MODEL,
    instructions=bot,
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "Summarise my flat-hunting notes in 3 lines."},
            {"type": "input_file", "file_id": notes_file.id},
        ],
    }],
)

print(response.output_text)

That works for one file. But with 6 PDFs (or 600!), sending **every** PDF with **every** question is slow and costly.

## 3. Way #2: a vector store 📚
A **vector store** cuts each PDF into small chunks and indexes them by **meaning**.
When you ask *"How much is the Bedok flat?"*, it finds the chunks about Bedok's price — even if the words don't match exactly.

First, make an empty library:

In [ ]:
vector_store = client.vector_stores.create(name="Flat hunting")

print(vector_store.id)

Now upload every PDF in `docs` into it. `upload_and_poll` waits until each file is ready to search.

In [ ]:
for name in os.listdir("docs"):
    client.vector_stores.files.upload_and_poll(
        vector_store_id=vector_store.id,
        file=open("docs/" + name, "rb"),
    )
    print("✅", name)

## 4. The file search tool 🔎
Like the python tool, **file search** is a built-in tool. We just tell it which library to search.

In [ ]:
search_tool = {"type": "file_search", "vector_store_ids": [vector_store.id]}

response = client.responses.create(
    model=MODEL,
    instructions=bot + " Use file search to answer from the user's documents.",
    tools=[search_tool],
    input="Which of the three listings is the cheapest, and who is the agent?",
)

print(response.output_text)

## 5. Which files did it use? 🧾
The reply comes with **citations** — notes saying which file each fact came from.

In [ ]:
for item in response.output:
    if item.type == "message":
        for citation in item.content[0].annotations:
            print("📄", citation.filename)

## 6. Documents + data 🤝
The PDFs say **what the agent is asking**. The CSV says **what flats really sold for**.
Give the AI both tools and let it check the agent's price!

In [ ]:
python_tool = {"type": "code_interpreter", "container": {"type": "auto", "file_ids": [os.environ["OPENAI_HDB_FILE_ID"]]}}   # saved in notebook 07

response = client.responses.create(
    model=MODEL,
    instructions=bot + " Use file search for the user's documents. Use the python tool to analyse hdb_resale_prices.csv.",
    tools=[search_tool, python_tool],
    input="Is the Tampines listing fairly priced compared to recent 4-room sales on the same street?",
)

print(response.output_text)

## 7. Save the library ID for your bot 💾

In [ ]:
with open(".env", "a") as f:
    f.write(f"\nOPENAI_VECTOR_STORE_ID={vector_store.id}\n")

print("Saved! Open .env to check.")

## ✏️ Exercises
1. Ask: *"Which renovation quote is cheaper once you add everything?"* Check the answer against `renovation_quotes.pdf` yourself!
2. Ask: *"Which flat did we like best at the viewings, and why?"*
3. Add your **own** PDF (a CV, a guide, a menu): drag it into `docs`, then upload just that one file with `upload_and_poll`.

In [ ]:
# your code here